# Geração de Dados para o Meta Model (Família IV)
**Objetivo:** Extrair as probabilidades contínuas (`predict_proba`) e predições de regressão dos modelos das Famílias I (Global X-Ray) e III (Mag Regional) operando sobre os conjuntos de Validação (Meta-Treino) e Teste cego.

Realizamos um `LEFT JOIN` ancorado no espaço-tempo regional (`T_REC_round`) para acoplar o contexto global (`time`) a cada mancha solar.

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from src.py_src.models import GatekeeperModel, GreatFilterModel, Specialist910Model, SpecialistMXModel

load_dotenv()

## 1. Carregamento dos Dados Brutos Stridados

In [ ]:
# Paths
SLIDED_PATH = os.getenv("SLIDED_PATH")
XRAY_PATH = os.path.join(SLIDED_PATH, "xray_slided.parquet")
MAG_REG_PATH = os.path.join(SLIDED_PATH, "mag_regional_slided.parquet")

# DataFrames
df_xray = pd.read_parquet(XRAY_PATH)
df_mag = pd.read_parquet(MAG_REG_PATH)

target_class = 'target_class_in_24h'
target_flux = 'target_flux_in_24h'

## 2. Funções de Particionamento (Blindadas)

In [ ]:
val_years = [2012, 2017]
test_years = [2020, 2021, 2022, 2023, 2024]

def extract_global_sets(df):
    """Extrai Val e Test para Global X-Ray com Purga Simétrica de 24h."""
    time_col = 'time'
    df = df.sort_values(time_col).reset_index(drop=True).copy()
    df['year'] = df[time_col].dt.year
    
    df['split'] = 'none'
    df.loc[df['year'].isin(val_years), 'split'] = 'val'
    df.loc[df['year'].isin(test_years), 'split'] = 'test'
    
    # Pontos de mudança para a purga
    df['block_change'] = df['split'] != df['split'].shift(1)
    df.loc[0, 'block_change'] = False
    
    drop_indices = set()
    change_indices = df[df['block_change']].index
    purge_td = pd.Timedelta(hours=24)
    
    for idx in change_indices:
        t_trans = df.loc[idx, time_col]
        to_drop = df[(df[time_col] >= t_trans - purge_td) & (df[time_col] < t_trans + purge_td)].index
        drop_indices.update(to_drop)
        
    df_purged = df.drop(index=list(drop_indices)).copy()
    
    df_val = df_purged[df_purged['split'] == 'val'].copy().reset_index(drop=True)
    df_test = df_purged[df_purged['split'] == 'test'].copy().reset_index(drop=True)
    
    return df_val, df_test

def extract_regional_sets(df):
    """Extrai Val e Test para Mag Regional baseado em HARP (Sem purga)."""
    time_col = 'T_REC_round'
    region_col = 'REGION_ID'
    
    df = df.sort_values([region_col, time_col]).reset_index(drop=True).copy()
    harp_birth = df.groupby(region_col)[time_col].min().dt.year.to_dict()
    df['harp_birth_year'] = df[region_col].map(harp_birth)
    
    df_val = df[df['harp_birth_year'].isin(val_years)].copy().reset_index(drop=True)
    df_test = df[df['harp_birth_year'].isin(test_years)].copy().reset_index(drop=True)
    
    return df_val, df_test

df_val_global, df_test_global = extract_global_sets(df_xray)
df_val_regional, df_test_regional = extract_regional_sets(df_mag)

print(f"Global: Val={len(df_val_global)}, Test={len(df_test_global)}")
print(f"Regional: Val={len(df_val_regional)}, Test={len(df_test_regional)}")

## 3. Inferência da Família I (Global X-Ray)

In [ ]:
BASE_PATH_I = os.getenv('GLOBAL_XRAY_FINAL_MODELS_PATH')

models_I = {
    'gk': GatekeeperModel.load(os.path.join(BASE_PATH_I, 'gatekeeper_v1.joblib')),
    'gf': GreatFilterModel.load(os.path.join(BASE_PATH_I, 'great_filter_v1.joblib')),
    's910': Specialist910Model.load(os.path.join(BASE_PATH_I, 'specialist_910_v1.joblib')),
    'smx': SpecialistMXModel.load(os.path.join(BASE_PATH_I, 'specialist_mx_v1.joblib'))
}

def generate_predictions_global(df):
    df_preds = pd.DataFrame({'time': df['time']})
    
    # 1. Realiza as predições de classes reais para simular a cascata
    pred_gk = models_I['gk'].predict(df)
    pred_gf = models_I['gf'].predict(df)

    # 2. Extrai as probabilidades originais
    prob_gk = models_I['gk'].predict_proba(df)[:, 1]
    prob_gf = models_I['gf'].predict_proba(df)[:, 1]
    prob_s910 = models_I['s910'].predict_proba(df)[:, 1]
    pred_smx = models_I['smx'].predict(df)

    # 3. MÁSCARAS DE CURTO-CIRCUITO (A essência da Cascata)
    # Especialistas profundos só ganham voz se a mancha sobreviver aos filtros anteriores
    mask_pass_gk = (pred_gk == 1)
    mask_pass_gf = mask_pass_gk & (pred_gf == 1)

    df_preds['prob_gk_I'] = prob_gk
    df_preds['prob_gf_I'] = np.where(mask_pass_gk, prob_gf, 0.0)
    df_preds['prob_s910_I'] = np.where(mask_pass_gf, prob_s910, 0.0)
    # Log flux basal muito baixo se não passou do Great Filter
    df_preds['pred_smx_I'] = np.where(mask_pass_gf, pred_smx, -8.0)
    
    # Features de contexto global úteis para o Meta Model
    df_preds['xrsb_flux_mean'] = df['xrsb_flux_mean']
    df_preds['xray_delta_6h'] = df['xray_delta_6h']
    
    return df_preds

preds_val_global = generate_predictions_global(df_val_global)
preds_test_global = generate_predictions_global(df_test_global)
print("Inferências Globais geradas com sucesso.")

## 4. Inferência da Família III (Mag Regional)

In [ ]:
BASE_PATH_III = os.getenv('REGIONAL_MAG_FINAL_MODELS_PATH')

models_III = {
    'gk': GatekeeperModel.load(os.path.join(BASE_PATH_III, 'gatekeeper_v1.joblib')),
    'gf': GreatFilterModel.load(os.path.join(BASE_PATH_III, 'great_filter_v1.joblib')),
    's910': Specialist910Model.load(os.path.join(BASE_PATH_III, 'specialist_910_v1.joblib')),
    'smx': SpecialistMXModel.load(os.path.join(BASE_PATH_III, 'specialist_mx_v1.joblib'))
}

def generate_predictions_regional(df):
    df_preds = pd.DataFrame({
        'T_REC_round': df['T_REC_round'],
        'REGION_ID': df['REGION_ID'],
        'target_class': df[target_class] # O Meta Model Regional avaliará a Mancha
    })

    # 1. Predições de classe
    pred_gk = models_III['gk'].predict(df)
    pred_gf = models_III['gf'].predict(df)

    # 2. Probabilidades brutas
    prob_gk = models_III['gk'].predict_proba(df)[:, 1]
    prob_gf = models_III['gf'].predict_proba(df)[:, 1]
    prob_s910 = models_III['s910'].predict_proba(df)[:, 1]
    pred_smx = models_III['smx'].predict(df)

    # 3. MÁSCARAS DE CURTO-CIRCUITO
    mask_pass_gk = (pred_gk == 1)
    mask_pass_gf = mask_pass_gk & (pred_gf == 1)

    df_preds['prob_gk_III'] = prob_gk
    df_preds['prob_gf_III'] = np.where(mask_pass_gk, prob_gf, 0.0)
    df_preds['prob_s910_III'] = np.where(mask_pass_gf, prob_s910, 0.0)
    df_preds['pred_smx_III'] = np.where(mask_pass_gf, pred_smx, -8.0)
    
    # Features de contexto regional
    df_preds['USFLUX'] = df['USFLUX']
    df_preds['AREA_ACR'] = df['AREA_ACR']
    df_preds['R_VALUE'] = df['R_VALUE']
    
    return df_preds

preds_val_regional = generate_predictions_regional(df_val_regional)
preds_test_regional = generate_predictions_regional(df_test_regional)
print("Inferências Regionais geradas com sucesso.")

## 5. O Grande JOIN (Alinhamento Espaço-Tempo)

In [ ]:
def construct_meta_dataset(df_regional, df_global):
    """
    Realiza o LEFT JOIN mantendo a granularidade regional.
    Uma mesma leitura de Raio-X Global será aplicada a todas as manchas ativas naquele minuto.
    """
    df_meta = pd.merge(
        df_regional,
        df_global,
        left_on='T_REC_round',
        right_on='time',
        how='inner' # INNER garante que só teremos dados com ambas as famílias disponíveis
    )
    
    # Remove a coluna de tempo duplicada
    df_meta = df_meta.drop(columns=['time'])
    
    # Feature Eng para o Meta Model: Contagem de manchas ativas no mesmo minuto
    spot_counts = df_meta.groupby('T_REC_round')['REGION_ID'].transform('count')
    df_meta['num_active_spots'] = spot_counts
    
    return df_meta

meta_val = construct_meta_dataset(preds_val_regional, preds_val_global)
meta_test = construct_meta_dataset(preds_test_regional, preds_test_global)

print(f"Meta-Treino (Validação Original): {len(meta_val)} amostras prontas.")
print(f"Meta-Teste (Teste Original): {len(meta_test)} amostras prontas.")

## 6. Exportação das Matrizes do Meta Model

In [ ]:
SAVE_DIR = os.path.join(os.getenv('SLIDED_PATH'), 'meta_model')
os.makedirs(SAVE_DIR, exist_ok=True)

meta_val.to_parquet(os.path.join(SAVE_DIR, 'meta_train.parquet'))
meta_test.to_parquet(os.path.join(SAVE_DIR, 'meta_test.parquet'))

print(f"Matrizes salvas com sucesso em: {SAVE_DIR}")
display(meta_val.head())